# Artificial Intelligence — Module 01
## Notebook 03: Intelligent Agents

Notebook by: **Caio Reis** (Repository Author)

### Learning Objectives

By the end of this notebook, you should be able to:

- Define an **Agent** and its **Environment**.
- Define **Rationality** in the context of agents.
- Describe a task environment using the **PEAS** framework.
- Classify different **Environment Types**.
- Differentiate between the main **Agent Architectures**.
- Implement a simple simulation for the **Vacuum-Cleaner World**.

> **Module Overview:**
> For a visual summary of all topics in this introductory module (including Knowledge Representation and Intelligent Agents), you can refer to the mind map.
> 
> You can find the source `.plantuml` file and the rendered `.png` image in the `artificial-intelligence/01-introduction/assets/` folder.

## 1. Agents and Environments

An **Agent** is anything that can be viewed as perceiving its **Environment** through **Sensors** and acting upon that environment through **Actuators**.

- **Human Agent:** 
  - *Sensors:* Eyes, ears, etc.
  - *Actuators:* Hands, legs, mouth.
- **Robotic Agent:** 
  - *Sensors:* Cameras, infrared detectors.
  - *Actuators:* Motors, arms.
- **Software Agent:**
  - *Sensors:* Keyboard input, network packets.
  - *Actuators:* Display on screen, send network packets.

The **Percept Sequence ($P*$)** is the complete history of everything the agent has ever perceived. The agent's behavior is defined by the **Agent Function ($f$)** which maps any given percept sequence to an action ($f: P* \rightarrow A$).

## 2. Rational Agents

A **Rational Agent** is one that, for each possible percept sequence, selects an action that is *expected* to maximize its **Performance Measure**, given the evidence provided by the percept sequence and any built-in knowledge the agent has.

This is a key distinction:

- **Rationality $\neq$ Perfection:** Rationality maximizes *expected* performance, while perfection maximizes *actual* performance. You can't be perfect in a partially observable world (e.g., you can't know if a card you don't see is an Ace), but you can play the odds rationally.
- **Information Gathering:** A rational agent may need to perform actions to gather information or explore (e.g., moving to an unknown room to see if it's dirty).
- **Autonomy:** An agent is **autonomous** if its behavior is determined by its own experience (learning) rather than just its initial programming.

## 3. PEAS Framework

To design an agent, we must first specify the **Task Environment**. We use the **PEAS** framework for this:

- **P**erformance Measure
- **E**nvironment
- **A**ctuators
- **S**ensors

### Exercise: Define PEAS

As an exercise, describe the PEAS for the following agents.

**1. Automated Taxi Driver:**
- **P:** (Performance) Safe, fast, legal, comfortable trip, maximize profits.
- **E:** (Environment) Roads, other vehicles, pedestrians, customers.
- **A:** (Actuators) Steering, accelerator, brake, signal, horn, display.
- **S:** (Sensors) Camera, sonar, speedometer, GPS, odometer, keyboard.

**2. Medical Diagnosis System:**
- **P:** Healthy patient, minimize costs, avoid lawsuits.
- **E:** Patient, hospital, staff.
- **A:** Display questions, tests, diagnoses, treatments.
- **S:** Keyboard entry of symptoms, findings, patient answers.

**3. Part-Picking Robot:**
- **P:** Percentage of parts in correct bins.
- **E:** Conveyor belt with parts; bins.
- **A:** Jointed arm and hand.
- **S:** Camera, joint angle sensors.

## 4. Environment Types

The environment type largely determines the agent design (e.g., as defined in Russell & Norvig).

| Property | Type 1 | Type 2 | Description |
| :--- | :--- | :--- | :--- |
| **Observability** | Fully Observable | Partially Observable | Do sensors give access to the *complete* state? |
| **Determinism** | Deterministic | Stochastic | Is the next state *completely* determined by the current state and action? |
| **Experience** | Episodic | Sequential | Is the agent's experience divided into atomic, independent episodes? |
| **Time** | Static | Dynamic | Can the environment change while the agent is deliberating? |
| **State Space** | Discrete | Continuous | Is the number of percepts and actions finite/countable? |
| **Agents** | Single-Agent | Multi-Agent | Is the agent operating by itself? |

**Example Environments:**

| Environment | Observable? | Deterministic? | Episodic? | Static? | Discrete? | Agents? |
| :--- | :--- | :--- | :--- | :--- | :--- | :--- |
| Chess | Yes | Yes | No (Sequential) | Yes | Yes | No (Multi-Agent) |
| Taxi Driving | No | No | No (Sequential) | No (Dynamic) | No (Continuous) | No (Multi-Agent) |
| Vacuum World | Yes | Yes | Yes | Yes | Yes | Yes (Single-Agent) |

## 5. Types of Agents

We can classify agents into several types, from simplest to most complex.

1.  **Simple Reflex Agents**
    - **Logic:** `if condition then action`
    - **How it works:** Reacts *only* to the current percept. Does not use any history. 
    - **Example:** `if state == Dirty then Suck`.
    - **Limitation:** Only works in fully observable environments. Can get stuck in loops.

2.  **Model-Based Reflex Agents**
    - **Logic:** Maintains an internal **state** (a *model* of the world).
    - **How it works:** Updates its internal state based on the percept and its knowledge of "how the world works" and "what its actions do". 
    - **Example:** A vacuum agent that remembers which rooms it has already cleaned.
    - **Limitation:** Can't make complex decisions about the future.

3.  **Goal-Based Agents**
    - **Logic:** Has a **goal** (a desirable state).
    - **How it works:** Considers the *future* impact of its actions. It asks: "Will this action lead me closer to my goal?" This often involves **search** and **planning** (which we will study in Module 2).
    - **Example:** A vacuum agent whose goal is "all rooms are clean". It will create a plan (e.g., go to Room B, then Room A) to achieve this.

4.  **Utility-Based Agents**
    - **Logic:** Has a **utility function** that defines "how happy" the agent is in a given state.
    - **How it works:** When there are multiple ways to achieve a goal (or multiple competing goals), the agent chooses the action that leads to the highest *expected utility*.
    - **Example:** A taxi agent choosing between a long, easy route and a short, high-traffic route. It weighs the utility of speed vs. safety vs. fuel consumption.

5.  **Learning Agents**
    - **Logic:** Can improve its performance over time by learning from experience.
    - **How it works:** Has a "performance element" (one of the agents above) and a "learning element" that modifies it. This is the basis for Machine Learning.

## 6. Mini Simulation — The Vacuum-Cleaner World

Let's implement a simple simulation of the 2-room Vacuum World. We will create:

1.  An `Environment` class that keeps track of dirt and agent position.
2.  A `SimpleReflexAgent` that only reacts to its current percept.
3.  A `ModelBasedReflexAgent` that *remembers* the state of the rooms it has seen.

In [ ]:
import random

class VacuumEnvironment:
    """A simple 2-room environment (A, B) for the vacuum cleaner.
       (Based on the 2-room vacuum world)
    """
    def __init__(self):
        # Environment state: Location A, Location B
        # State can be 'Clean' or 'Dirty'
        self.rooms = {"A": "Clean", "B": "Clean"}
        # Randomly make rooms dirty
        if random.random() < 0.5:
            self.rooms["A"] = "Dirty"
        if random.random() < 0.5:
            self.rooms["B"] = "Dirty"
            
        self.agent_location = "A" # Agent starts in A
        self.performance_score = 0
        self.log = []

    def get_percept(self):
        """Returns the agent's percept: (location, status)"""
        return (self.agent_location, self.rooms[self.agent_location])

    def step(self, action):
        """Executes the agent's action and updates the environment.
           Actions: 'Suck', 'Left', 'Right', 'NoOp'
           (Loses 1 point per action)
        """
        if action == "Suck":
            if self.rooms[self.agent_location] == "Dirty":
                self.rooms[self.agent_location] = "Clean"
                self.performance_score += 10 # +10 for cleaning dirt
            self.performance_score -= 1 # -1 for action cost
            self.log.append(f"Percept: {self.get_percept()} -> Action: Suck")
            
        elif action == "Right":
            self.agent_location = "B"
            self.performance_score -= 1
            self.log.append(f"Percept: {self.get_percept()} -> Action: Right")
            
        elif action == "Left":
            self.agent_location = "A"
            self.performance_score -= 1
            self.log.append(f"Percept: {self.get_percept()} -> Action: Left")
        
        elif action == "NoOp":
            self.performance_score -= 1
            self.log.append(f"Percept: {self.get_percept()} -> Action: NoOp")
            
        return self.get_percept()

    def print_log(self):
        print("--- Simulation Log ---")
        for entry in self.log:
            print(entry)
        print(f"Final State: {self.rooms}")
        print(f"Final Score: {self.performance_score}")

# --- Agent Definitions ---

def SimpleReflexAgent(percept):
    """This agent only reacts to the current percept."""
    location, status = percept
    if status == "Dirty":
        return "Suck"
    elif location == "A":
        return "Right"
    elif location == "B":
        return "Left"

class ModelBasedReflexAgent:
    """This agent maintains an internal state (model) of the world."""
    def __init__(self):
        # Model: Agent's *belief* about the world. 
        # 'None' means 'unknown'
        self.model = {"A": None, "B": None}
        
    def program(self, percept):
        location, status = percept
        # Update internal model based on percept
        self.model[location] = status
        
        # Now, act based on the *model*
        if self.model[location] == "Dirty":
            return "Suck"
        
        # Current location is clean, check the other one
        other_loc = "B" if location == "A" else "A"
        if self.model[other_loc] != "Clean": # Go to other room if unknown or dirty
            return "Right" if location == "A" else "Left"
        
        # Both rooms are clean in our model, so do nothing.
        return "NoOp" 

# --- Run Simulation ---

print("=== RUN 1: Simple Reflex Agent ===")
random.seed(42) # Set a seed for consistent results
env_simple = VacuumEnvironment()
agent_simple_program = SimpleReflexAgent

print(f"Initial State: {env_simple.rooms}, Agent at: {env_simple.agent_location}")
for _ in range(8): # Run for 8 steps
    current_percept = env_simple.get_percept()
    action = agent_simple_program(current_percept)
    env_simple.step(action)

env_simple.print_log()

print("\n=== RUN 2: Model-Based Agent ===")
random.seed(42) # Use the *same* starting conditions
env_model = VacuumEnvironment()
agent_model = ModelBasedReflexAgent()

print(f"Initial State: {env_model.rooms}, Agent at: {env_model.agent_location}")
for _ in range(8):
    current_percept = env_model.get_percept()
    action = agent_model.program(current_percept)
    env_model.step(action)

env_model.print_log()

### Reflection

Look at the simulation results. 

1.  **Simple Reflex Agent:** Notice that even after it cleans both rooms, it continues to move `Left` and `Right` forever. Why? Because its logic is `if Clean then Move`. It has no memory and can't know that *both* rooms are clean. This is an **infinite loop**.
2.  **Model-Based Agent:** Notice that after it cleans both rooms (e.g., `Suck` in A, `Right`, `Suck` in B), it updates its model to `{'A': 'Clean', 'B': 'Clean'}`. Its logic then tells it to do `NoOp` (No Operation). 

**Question:** Which agent is more rational given the performance measure (`+10` for dirt, `-1` per action)? Why?